# Benchmark of LLM agent performance in uncertain environments
This notebook analyzes the performance of an llm trying to navigate an uncertain environment. Using a lookahead search it decides which steps are the best possible moves. To value each step another llm is leveraged with a list of hypothese and policies about the environment.


In [ ]:
# Setup + Imports
from evaluation.Benchmark import Benchmark
from openai import OpenAI
import gymnasium as gym
from tinydb import TinyDB
import os
from navigation.environments.FrozenLakeEnv import FrozenLakeEnv
from navigation.environments.FrozenLakeShadowEnv import FrozenLakeShadowEnv
from optimization.prompts.FrozenLakePrompts import FrozenLakePrompts

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key= os.getenv("OPENROUTER_API_KEY")
)
model = "openai/gpt-oss-120b"

## FrozenLake Benchmarks

In [ ]:
policyDb = TinyDB("../store/policies.json")
hypothesesDb = TinyDB("../store/hypotheses.json")

env = FrozenLakeEnv(env = gym.make("FrozenLake-v1", render_mode="ansi", desc=None, map_name="4x4", is_slippery=False, success_rate=0.7, reward_schedule=(1, 0, 0)))
shadow_env = FrozenLakeShadowEnv(hypothesesDb=hypothesesDb, policyDb=policyDb, client=client, model=model)
optimizationPrompts = FrozenLakePrompts(policyDb=policyDb, hypothesesDb=hypothesesDb)

bench_smallMap_noLlm = Benchmark(env=env, shadow_env=shadow_env, policyDb=policyDb, hypothesesDb=hypothesesDb, optimizationPrompts=optimizationPrompts, client=client, model=model)
policyDb.truncate()
hypothesesDb.truncate()

In [ ]:
# no lookahead, no llm action, small map
metrics_nlook_nllm_small = bench_smallMap_noLlm.run(name="frozenlake_small", iteration_depth=1, lookahead_sample_size=1, lookahead_depth=1, max_nav_steps=30, use_llm_action=False, print_debug=True)
policyDb.truncate()
hypothesesDb.truncate()

In [ ]:
# no lookahead, llm action, small map
metrics_nlook_llm_small = bench_smallMap_noLlm.run(name="frozenlake_small", iteration_depth=10, lookahead_sample_size=1, lookahead_depth=1, max_nav_steps=30, use_llm_action=True, print_debug=True)
policyDb.truncate()
hypothesesDb.truncate()

In [ ]:
# lookahead, no llm action, small map
metrics_2look_nllm_small = bench_smallMap_noLlm.run(name="frozenlake_small", iteration_depth=10, lookahead_sample_size=2, lookahead_depth=2, max_nav_steps=30, use_llm_action=False, print_debug=True)
policyDb.truncate()
hypothesesDb.truncate()

In [ ]:
# lookahead, llm action, small map
metrics_2look_llm_small = bench_smallMap_noLlm.run(name="frozenlake_small", iteration_depth=10, lookahead_sample_size=2, lookahead_depth=2, max_nav_steps=30, use_llm_action=True, print_debug=True)
policyDb.truncate()
hypothesesDb.truncate()